# GCMC Hybrid Run Analysis
Load and visualise trajectory data from `mu_scan_gcmc.py` runs.

In [ ]:
import os
import glob
import re

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 10, 'figure.dpi': 120})

## 1. Discover and load mu-scan NPZ files

In [ ]:
MU_SCAN_DIR = 'data/mu_scan'

def parse_mu_scan_dir(data_dir):
    """Return list of dicts, one per NPZ, with parsed params + trajectory arrays."""
    runs = []
    for path in sorted(glob.glob(os.path.join(data_dir, '*.npz'))):
        d = np.load(path)
        runs.append(dict(
            path    = path,
            mu      = float(d['mu'][0]),
            EAB     = float(d['E_AB'][0]),
            oa_deg  = float(np.degrees(d['opening_angle'][0])),
            L       = float(np.sqrt(d['box_area'][0])),
            kT      = float(d['kT'][0]),
            n_prod  = int(d['n_prod'][0]),
            N_traj  = d['N_traj'].copy(),   # (n_prod,) int32  — every sweep
            E_traj  = d['E_traj'].copy(),   # (n_snapshots,) float32
            acc_ins = float(d['acc_ins'][0]),
            acc_del = float(d['acc_del'][0]),
        ))
    return runs

runs = parse_mu_scan_dir(MU_SCAN_DIR)
print(f'Loaded {len(runs)} mu-scan runs.')
print(f'  EAB values : {sorted(set(r["EAB"]   for r in runs))}')
print(f'  mu  values : {sorted(set(r["mu"]    for r in runs))}')
print(f'  oa  values : {sorted(set(r["oa_deg"] for r in runs))}')

## 2. N(t) and E(t) — vary opening angle at fixed E_AB and μ

In [ ]:
TARGET_EAB = 5.0
TARGET_MU  = -3.0
TARGET_OAS = [75.0, 90.0, 105.0, 120.0]   # deg; edit to taste

selected = [
    r for r in runs
    if abs(r['EAB'] - TARGET_EAB) < 0.01
    and abs(r['mu']  - TARGET_MU)  < 0.01
    and any(abs(r['oa_deg'] - oa) < 0.5 for oa in TARGET_OAS)
]
selected.sort(key=lambda r: r['oa_deg'])

print(f'{len(selected)} runs selected:')
for r in selected:
    print(f"  oa={r['oa_deg']:.0f}°  <N>={r['N_traj'].mean():.1f}±{r['N_traj'].std():.1f}")

cmap   = plt.cm.viridis
colors = [cmap(i / max(len(selected) - 1, 1)) for i in range(len(selected))]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax_N, ax_E = axes

for r, c in zip(selected, colors):
    label    = f"oa={r['oa_deg']:.0f}°"
    sweeps_N = np.arange(1, len(r['N_traj']) + 1)
    ax_N.plot(sweeps_N, r['N_traj'], lw=0.8, alpha=0.85, color=c, label=label)

    if len(r['E_traj']) > 0:
        n_snap   = len(r['E_traj'])
        interval = r['n_prod'] // n_snap
        sweeps_E = np.arange(1, n_snap + 1) * interval
        N_at_E   = r['N_traj'][sweeps_E - 1]
        ax_E.plot(sweeps_E, r['E_traj'] / np.maximum(N_at_E, 1),
                  lw=1.2, marker='o', ms=3, color=c, label=label)

ax_N.set_xlabel('Production sweep')
ax_N.set_ylabel('N particles')
ax_N.set_title(f'Particle count  (EAB={TARGET_EAB}, μ={TARGET_MU})')
ax_N.legend(fontsize=8, ncol=2)
ax_N.grid(True, lw=0.4, alpha=0.4)

ax_E.set_xlabel('Production sweep')
ax_E.set_ylabel('E / N  (kT)')
ax_E.set_title(f'Energy per particle  (EAB={TARGET_EAB}, μ={TARGET_MU})')
ax_E.legend(fontsize=8, ncol=2)
ax_E.grid(True, lw=0.4, alpha=0.4)

fig.tight_layout()
plt.show()

## 3. N(t) and E(t) — vary E_AB at fixed opening angle and μ

In [ ]:
TARGET_OA  = 90.0
TARGET_MU2 = -3.0

sel2 = [
    r for r in runs
    if abs(r['oa_deg'] - TARGET_OA) < 0.5
    and abs(r['mu'] - TARGET_MU2) < 0.01
]
sel2.sort(key=lambda r: r['EAB'])

colors2 = [cmap(i / max(len(sel2) - 1, 1)) for i in range(len(sel2))]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax_N2, ax_E2 = axes

for r, c in zip(sel2, colors2):
    label    = f"EAB={r['EAB']:.0f}"
    sweeps_N = np.arange(1, len(r['N_traj']) + 1)
    ax_N2.plot(sweeps_N, r['N_traj'], lw=0.8, alpha=0.85, color=c, label=label)

    if len(r['E_traj']) > 0:
        n_snap   = len(r['E_traj'])
        interval = r['n_prod'] // n_snap
        sweeps_E = np.arange(1, n_snap + 1) * interval
        N_at_E   = r['N_traj'][sweeps_E - 1]
        ax_E2.plot(sweeps_E, r['E_traj'] / np.maximum(N_at_E, 1),
                   lw=1.2, marker='o', ms=3, color=c, label=label)

ax_N2.set_xlabel('Production sweep')
ax_N2.set_ylabel('N particles')
ax_N2.set_title(f'Particle count  (oa={TARGET_OA:.0f}°, μ={TARGET_MU2})')
ax_N2.legend(fontsize=8, ncol=2)
ax_N2.grid(True, lw=0.4, alpha=0.4)

ax_E2.set_xlabel('Production sweep')
ax_E2.set_ylabel('E / N  (kT)')
ax_E2.set_title(f'Energy per particle  (oa={TARGET_OA:.0f}°, μ={TARGET_MU2})')
ax_E2.legend(fontsize=8, ncol=2)
ax_E2.grid(True, lw=0.4, alpha=0.4)

fig.tight_layout()
plt.show()

## 4. Summary table

In [ ]:
print(f"{'mu':>7}  {'EAB':>6}  {'oa(deg)':>8}  {'<N>':>6}  {'std_N':>6}  {'acc_ins':>8}  {'acc_del':>8}")
print('-' * 66)
for r in sorted(runs, key=lambda r: (r['mu'], r['EAB'], r['oa_deg'])):
    print(
        f"{r['mu']:7.4f}  {r['EAB']:6.1f}  {r['oa_deg']:8.2f}  "
        f"{r['N_traj'].mean():6.1f}  {r['N_traj'].std():6.2f}  "
        f"{r['acc_ins']:8.4f}  {r['acc_del']:8.4f}"
    )